"""RvC Mechanistic Sweep v4 — Clean rewrite.

DELIVERABLES (what the CSV must contain for the paper):
  - crystallization_layer : int
        Layer at which the correct first-action token first becomes
        the model's top-1 logit-lens prediction.
        -1 = never crystallized across all 24 layers.
        Paper claim: canonical problems = all -1 (complete non-crystallization).
        W6 problems = hopefully some positive value (early crystallization = retrieval).
  - layer_cosine_similarities : JSON list of 24 floats
        Per-layer cosine similarity between canonical residual stream
        and each W2/W3/W4 variant residual stream, averaged across variants.
        Used to measure surface invariance at the representation level.
  - problem_id, problem_family, variant_type, model, n_layers_processed : metadata

STRATEGY:
  This notebook does NOT call run_mechanistic_sweep.py as a subprocess.
  All sweep logic runs directly in the notebook cells.
  The model is loaded ONCE and kept alive across all problems.
  RAM is freed after each problem by deleting the cache tensor immediately.

RUN ORDER (one cell at a time, wait for checkmark):
  Cell 1 → Cell 2 → Cell 3 → Cell 4 → Cell 5 → Cell 6 →
  Cell 7 (canonical) → Cell 8 (W6) → Cell 9 (inspect) → Cell 10 (download)

Paste keepalive in browser console (F12 → Console):
  function KeepAlive(){document.querySelector('colab-connect-button').click();
  setTimeout(KeepAlive,60000);}KeepAlive();
"""

In [1]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM total: {total:.1f} GB')
    print(f'VRAM free:  {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')
else:
    print('NO GPU — Runtime > Change runtime type > T4 GPU')
    raise RuntimeError('GPU required')


GPU available: True
GPU: Tesla T4
VRAM total: 15.6 GB
VRAM free:  15.5 GB


In [2]:
# ─────────────────────────────────────────────
# CELL 2 — Clone repo and install deps
# ─────────────────────────────────────────────
import os, subprocess

REPO = '/content/retrieval-vs-computation'
if not os.path.exists(REPO):
    os.system('git clone https://github.com/Adya6714/retrieval-vs-computation.git')
    print('Cloned.')
else:
    print('Repo already exists.')

os.chdir(REPO)
print('cwd:', os.getcwd())

# Uncomment mechanistic deps
with open('requirements.txt', 'r') as f:
    req = f.read()
for dep in ['# transformer_lens>=1.19', '# transformers>=4.41',
            '# accelerate>=0.30', '# bitsandbytes>=0.43']:
    req = req.replace(dep, dep.lstrip('# '))
with open('requirements.txt', 'w') as f:
    f.write(req)

print('Installing deps (~3 min)...')
r = subprocess.run(['pip', 'install', '-r', 'requirements.txt', '-q'],
                   capture_output=True, text=True)
if r.returncode != 0:
    print('INSTALL ERROR:', r.stderr[-2000:])
else:
    print('Base install OK.')

r2 = subprocess.run(['pip', 'install', '--upgrade', 'transformer_lens', '-q'],
                    capture_output=True, text=True)
print('transformer_lens upgrade:', 'OK' if r2.returncode == 0 else r2.stderr[-500:])

from transformer_lens import HookedTransformer
print('transformer_lens import: OK')


Repo already exists.
cwd: /content/retrieval-vs-computation
Installing deps (~3 min)...
Base install OK.
transformer_lens upgrade: OK
transformer_lens import: OK


In [3]:
# ─────────────────────────────────────────────
# CELL 3 — Build variants lookup table
#
# Reads question_bank.csv and builds two dicts:
#   VARIANTS[problem_id] = list of (variant_type, variant_text) for W2/W3/W4
#   W6_TEXT[problem_id]  = variant_text for W6
# Also builds CORRECT_FIRST_ACTION[problem_id] = first move string
# ─────────────────────────────────────────────
import pandas as pd, json, os

qb = pd.read_csv('data/problems/question_bank.csv')
print(f'question_bank.csv: {len(qb)} rows')
print('variant_type counts:')
print(qb['variant_type'].value_counts())

# Canonical problems only
canonical_df = qb[qb['variant_type'] == 'canonical'].copy()
canonical_ids = canonical_df['problem_id'].tolist()
print(f'\nCanonical problems: {canonical_ids}')

# Build lookup: base_id → list of (variant_type, problem_text)
VARIANTS = {}   # W2/W3/W4 variants for cosine sim
W6_TEXT  = {}   # W6 variant text for crystallization comparison

for _, row in qb.iterrows():
    pid = str(row['problem_id']).strip()
    vtype = str(row['variant_type']).strip()

    if vtype in ('W2', 'W3', 'W4'):
        # Strip _W2/_W3/_W4 suffix to get base_id
        import re
        base = re.sub(r'_(W[2-4])$', '', pid)
        if base not in VARIANTS:
            VARIANTS[base] = []
        VARIANTS[base].append((vtype, str(row['problem_text'])))

    elif vtype == 'W6':
        base = re.sub(r'_W6$', '', pid)
        W6_TEXT[base] = str(row['problem_text'])

# Build correct first action lookup from correct_answer column
CORRECT_FIRST_ACTION = {}
for _, row in canonical_df.iterrows():
    pid = str(row['problem_id']).strip()
    answer = str(row['correct_answer']).strip()
    # First action is the first non-empty line
    lines = [l.strip() for l in answer.split('\n') if l.strip()]
    CORRECT_FIRST_ACTION[pid] = lines[0] if lines else ''

print('\nFirst actions:')
for pid, action in CORRECT_FIRST_ACTION.items():
    variants_count = len(VARIANTS.get(pid, []))
    has_w6 = pid in W6_TEXT
    print(f'  {pid}: "{action}" | variants: {variants_count} | W6: {has_w6}')

print('\nCell 3 OK.')

question_bank.csv: 140 rows
variant_type counts:
variant_type
canonical    21
W1           21
W2           21
W3           21
W4           21
W5           20
W6           15
Name: count, dtype: int64

Canonical problems: ['BW_155', 'BW_010', 'BW_001', 'BW_011', 'BW_120', 'BW_227', 'BW_137', 'BW_172', 'BW_467', 'BW_080', 'BW_E002', 'BW_E015', 'BW_E017', 'BW_E019', 'BW_E100', 'MBW_001', 'MBW_100', 'MBW_127', 'MBW_185', 'MBW_10', 'MBW_10']

First actions:
  BW_155: "pick-up j" | variants: 3 | W6: True
  BW_010: "pick-up b" | variants: 3 | W6: True
  BW_001: "pick-up h" | variants: 3 | W6: True
  BW_011: "pick-up i" | variants: 3 | W6: True
  BW_120: "pick-up g" | variants: 3 | W6: True
  BW_227: "pick-up k" | variants: 3 | W6: True
  BW_137: "pick-up g" | variants: 3 | W6: True
  BW_172: "pick-up f" | variants: 3 | W6: True
  BW_467: "pick-up f" | variants: 3 | W6: True
  BW_080: "pick-up j" | variants: 3 | W6: True
  BW_E002: "pick-up a" | variants: 3 | W6: True
  BW_E015: "unstack c b" 

In [4]:
# ─────────────────────────────────────────────
# CELL 4 — Load model ONCE
#
# Model stays in memory for Cells 7 and 8.
# Do NOT re-run this cell after the model is loaded.
# ─────────────────────────────────────────────
import gc, torch
from transformer_lens import HookedTransformer
from transformers import AutoTokenizer

gc.collect()
torch.cuda.empty_cache()
print(f'VRAM free before load: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, add_bos_token=False)

model = HookedTransformer.from_pretrained(
    MODEL_NAME,
    tokenizer=tokenizer,
    device='cuda',
    dtype=torch.float16,
    default_prepend_bos=False,
    fold_ln=False,
    center_writing_weights=False,
    center_unembed=False,
)

print(f'Model loaded: {MODEL_NAME}')
print(f'Layers: {model.cfg.n_layers} | d_model: {model.cfg.d_model}')
print(f'VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB')
print(f'VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

# Quick sanity check
_tok = model.to_tokens('pick-up a', prepend_bos=False)
_logits, _cache = model.run_with_cache(_tok)
_resid_keys = [k for k in _cache.keys() if 'resid_post' in k]
print(f'\nSanity check: {len(_resid_keys)} residual stream keys')
print(f'Key format: {_resid_keys[0]}')
print(f'Activation shape: {_cache[_resid_keys[0]].shape}')
del _logits, _cache, _tok
torch.cuda.empty_cache()
print('\nCell 4 OK — model ready.')

VRAM free before load: 15.5 GB


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B-Instruct into HookedTransformer
Model loaded: Qwen/Qwen2.5-0.5B-Instruct
Layers: 24 | d_model: 896
VRAM used: 1.4 GB
VRAM free: 14.1 GB

Sanity check: 24 residual stream keys
Key format: blocks.0.hook_resid_post
Activation shape: torch.Size([1, 3, 896])

Cell 4 OK — model ready.


In [5]:
# ─────────────────────────────────────────────
# CELL 5 — Core sweep functions
#
# get_crystallization_layer(problem_text, correct_first_action)
#   → int  (-1 if never top-1 across all layers)
#
# get_cosine_sims(canonical_text, variant_texts)
#   → list of 24 floats (per-layer mean cosine sim)
#
# These run entirely inside the notebook.
# No subprocess. No W1 logic. No skip conditions.
# ─────────────────────────────────────────────
import torch, gc
import torch.nn.functional as F

def _get_last_token_resid(text):
    """
    Run a forward pass and return residual stream at the last token
    position for every layer. Returns tensor of shape (n_layers, d_model).
    Cache is deleted immediately after extraction.
    """
    tokens = model.to_tokens(text, prepend_bos=False)
    with torch.no_grad():
        logits, cache = model.run_with_cache(tokens)

    n_layers = model.cfg.n_layers
    resid_keys = [f'blocks.{i}.hook_resid_post' for i in range(n_layers)]

    # Extract last token residual for each layer — shape (n_layers, d_model)
    acts = torch.stack(
        [cache[k][0, -1, :].float() for k in resid_keys]
    )  # (n_layers, d_model)

    # Free immediately
    del logits, cache
    torch.cuda.empty_cache()
    gc.collect()

    return acts  # stays on GPU as float32


def get_crystallization_layer(problem_text, correct_first_action):
    """
    Check crystallization for the first FULL action by checking
    the top-5 tokens at each layer (not just top-1), and also check
    the first token of the action without numbering prefix.
    Returns first layer where target is in top-5. -1 if never.
    """
    if not correct_first_action.strip():
        print('  WARNING: empty correct_first_action')
        return -1

    # Strip any leading number like "1. " or "1) "
    import re
    action_clean = re.sub(r'^\d+[\.\)]\s*', '', correct_first_action.strip())

    # Get ALL token ids for the full action to check against
    target_tokens = model.to_tokens(action_clean, prepend_bos=False)
    target_id_first = target_tokens[0, 0].item()  # first sub-token of action
    target_str = model.tokenizer.decode([target_id_first])
    print(f'  Target first token: {repr(target_str)} (id={target_id_first})')

    acts = _get_last_token_resid(problem_text)  # (n_layers, d_model) float32

    n_layers = acts.shape[0]
    W_U = model.unembed.W_U.float()
    b_U = model.unembed.b_U.float()

    crystallization_layer = -1
    top5_at_each_layer = []

    for layer_i in range(n_layers):
        layer_logits = acts[layer_i] @ W_U + b_U
        top5 = layer_logits.topk(5).indices.tolist()
        top5_tokens = [model.tokenizer.decode([t]) for t in top5]
        top5_at_each_layer.append(top5_tokens)

        if target_id_first in top5:
            crystallization_layer = layer_i
            print(f'  Crystallized at layer {layer_i} (top-5) | top5: {top5_tokens}')
            break

    if crystallization_layer == -1:
        # Print last layer top5 for debugging
        print(f'  Never crystallized. Last layer top-5: {top5_at_each_layer[-1]}')
        # Also print layer where target got closest (highest rank)
        all_logits_at_target = []
        for layer_i in range(n_layers):
            layer_logits = acts[layer_i] @ W_U + b_U
            rank = (layer_logits > layer_logits[target_id_first]).sum().item()
            all_logits_at_target.append(rank)
        best_layer = min(range(n_layers), key=lambda i: all_logits_at_target[i])
        print(f'  Best layer for target: layer {best_layer}, rank {all_logits_at_target[best_layer]}')

    del acts, W_U, b_U
    torch.cuda.empty_cache()
    gc.collect()

    return crystallization_layer

def get_cosine_sims(canonical_text, variant_texts):
    """
    For each variant, compute per-layer cosine similarity of the last-token
    residual stream between canonical and variant.
    Returns a list of 24 floats (mean across all variants per layer).
    Returns [] if no variants provided.
    """
    if not variant_texts:
        return []

    canon_acts = _get_last_token_resid(canonical_text)  # (n_layers, d_model)
    n_layers = canon_acts.shape[0]

    all_sims = []  # list of (n_layers,) tensors, one per variant
    for vtext in variant_texts:
        var_acts = _get_last_token_resid(vtext)  # (n_layers, d_model)
        # Cosine similarity per layer
        sims = F.cosine_similarity(
            canon_acts,   # (n_layers, d_model)
            var_acts,     # (n_layers, d_model)
            dim=1         # over d_model → result shape (n_layers,)
        )
        all_sims.append(sims)
        del var_acts
        gc.collect()

    del canon_acts
    torch.cuda.empty_cache()
    gc.collect()

    # Mean across variants per layer
    mean_sims = torch.stack(all_sims).mean(dim=0)  # (n_layers,)
    return [round(float(x), 4) for x in mean_sims.tolist()]


print('Cell 5 OK — sweep functions defined.')
print('  get_crystallization_layer(problem_text, correct_first_action) → int')
print('  get_cosine_sims(canonical_text, variant_texts) → list[float]')


Cell 5 OK — sweep functions defined.
  get_crystallization_layer(problem_text, correct_first_action) → int
  get_cosine_sims(canonical_text, variant_texts) → list[float]


In [6]:
# ─────────────────────────────────────────────
# CELL 6 — Test functions on one problem
#
# Runs BW_001 as a quick check before the full sweep.
# Should complete in ~30 seconds.
# ─────────────────────────────────────────────
import pandas as pd, json

qb = pd.read_csv('data/problems/question_bank.csv')

# Pick first canonical problem
test_pid = canonical_ids[0]
test_row = qb[(qb['problem_id'] == test_pid) & (qb['variant_type'] == 'canonical')].iloc[0]
test_text = str(test_row['problem_text'])
test_action = CORRECT_FIRST_ACTION[test_pid]

print(f'Test problem: {test_pid}')
print(f'First action: "{test_action}"')
print(f'Text length: {len(test_text)} chars')

# Test crystallization
print('\nRunning crystallization test...')
cl = get_crystallization_layer(test_text, test_action)
print(f'crystallization_layer = {cl}  (expected: -1 for hard BW problems)')

# Test cosine sims
variant_texts = [vt for _, vt in VARIANTS.get(test_pid, [])]
print(f'\nRunning cosine sim test ({len(variant_texts)} variants)...')
sims = get_cosine_sims(test_text, variant_texts)
if sims:
    print(f'cosine sims: {len(sims)} layers | min={min(sims):.3f} max={max(sims):.3f} mean={sum(sims)/len(sims):.3f}')
else:
    print('No variants found for this problem — sims = []')

print(f'\nVRAM free after test: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')
print('\nCell 6 OK — functions working. Proceed to Cell 7.')

Test problem: BW_155
First action: "pick-up j"
Text length: 663 chars

Running crystallization test...
  Target first token: 'pick' (id=29245)
  Never crystallized. Last layer top-5: [' ', '.\n\n', '.\n', ' Robot', ' Based']
  Best layer for target: layer 23, rank 1062
crystallization_layer = -1  (expected: -1 for hard BW problems)

Running cosine sim test (3 variants)...
cosine sims: 24 layers | min=0.537 max=0.798 mean=0.705

VRAM free after test: 14.0 GB

Cell 6 OK — functions working. Proceed to Cell 7.


In [7]:
# ─────────────────────────────────────────────
# CELL 7 — Canonical sweep
#
# Runs all canonical BW problems.
# Saves results to results/probe1_mechanistic.csv after EACH problem.
# Safe to interrupt and resume — already-done problems are skipped.
# ─────────────────────────────────────────────
import pandas as pd, json, os, gc, torch, time

CSV_PATH = 'results/probe1_mechanistic.csv'
os.makedirs('results', exist_ok=True)

# Define columns
COLS = ['problem_id', 'problem_family', 'variant_type', 'model',
        'crystallization_layer', 'n_layers_processed', 'layer_cosine_similarities']

# Load existing results for resume
if os.path.exists(CSV_PATH):
    df_existing = pd.read_csv(CSV_PATH)
    # Drop any GPT-2 artifact rows
    df_existing = df_existing[df_existing['model'] != 'gpt2']
    done_ids = set(df_existing['problem_id'].tolist())
    print(f'Resuming — {len(done_ids)} already done: {done_ids}')
else:
    df_existing = pd.DataFrame(columns=COLS)
    done_ids = set()
    print('Starting fresh.')

qb = pd.read_csv('data/problems/question_bank.csv')
canonical_df = qb[qb['variant_type'] == 'canonical'].copy()

print(f'\nProblems to run: {canonical_ids}')
print(f'Already done: {done_ids}')
print(f'Remaining: {[p for p in canonical_ids if p not in done_ids]}\n')

results = df_existing.to_dict('records')

for i, pid in enumerate(canonical_ids):
    if pid in done_ids:
        print(f'[{i+1}/{len(canonical_ids)}] {pid} — skipping (already done)')
        continue

    gc.collect()
    torch.cuda.empty_cache()
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    print(f'[{i+1}/{len(canonical_ids)}] {pid} | VRAM free: {free_gb:.1f} GB')

    row = canonical_df[canonical_df['problem_id'] == pid].iloc[0]
    problem_text = str(row['problem_text'])
    family = str(row['problem_family'])
    first_action = CORRECT_FIRST_ACTION.get(pid, '')

    # Step 1: Crystallization layer
    print(f'  Running crystallization (target: "{first_action[:40]}")...')
    try:
        cl = get_crystallization_layer(problem_text, first_action)
        print(f'  crystallization_layer = {cl}')
    except Exception as e:
        print(f'  crystallization FAILED: {e}')
        cl = -1

    # Step 2: Cosine similarities with W2/W3/W4 variants
    variant_list = VARIANTS.get(pid, [])
    variant_texts = [vt for _, vt in variant_list]
    print(f'  Running cosine sims ({len(variant_texts)} variants)...')
    try:
        sims = get_cosine_sims(problem_text, variant_texts)
        if sims:
            print(f'  sims: min={min(sims):.3f} max={max(sims):.3f} mean={sum(sims)/len(sims):.3f}')
        else:
            print('  sims: [] (no W2/W3/W4 variants in question_bank)')
    except Exception as e:
        print(f'  cosine sims FAILED: {e}')
        sims = []

    # Save row
    result_row = {
        'problem_id': pid,
        'problem_family': family,
        'variant_type': 'canonical',
        'model': 'Qwen/Qwen2.5-0.5B-Instruct',
        'crystallization_layer': cl,
        'n_layers_processed': model.cfg.n_layers,
        'layer_cosine_similarities': json.dumps(sims),
    }
    results.append(result_row)
    done_ids.add(pid)

    # Write to CSV after every single problem — safe against crashes
    pd.DataFrame(results, columns=COLS).to_csv(CSV_PATH, index=False)
    print(f'  Saved. ({len(results)} total rows in CSV)')

    time.sleep(1)  # brief pause to let GPU cool

print(f'\n=== Cell 7 complete: {len(results)}/{len(canonical_ids)} canonical problems ===')
df_check = pd.read_csv(CSV_PATH)
print(df_check[['problem_id', 'crystallization_layer', 'n_layers_processed']].to_string())


Starting fresh.

Problems to run: ['BW_155', 'BW_010', 'BW_001', 'BW_011', 'BW_120', 'BW_227', 'BW_137', 'BW_172', 'BW_467', 'BW_080', 'BW_E002', 'BW_E015', 'BW_E017', 'BW_E019', 'BW_E100', 'MBW_001', 'MBW_100', 'MBW_127', 'MBW_185', 'MBW_10', 'MBW_10']
Already done: set()
Remaining: ['BW_155', 'BW_010', 'BW_001', 'BW_011', 'BW_120', 'BW_227', 'BW_137', 'BW_172', 'BW_467', 'BW_080', 'BW_E002', 'BW_E015', 'BW_E017', 'BW_E019', 'BW_E100', 'MBW_001', 'MBW_100', 'MBW_127', 'MBW_185', 'MBW_10', 'MBW_10']

[1/21] BW_155 | VRAM free: 14.0 GB
  Running crystallization (target: "pick-up j")...
  Target first token: 'pick' (id=29245)
  Never crystallized. Last layer top-5: [' ', '.\n\n', '.\n', ' Robot', ' Based']
  Best layer for target: layer 23, rank 1062
  crystallization_layer = -1
  Running cosine sims (3 variants)...
  sims: min=0.537 max=0.798 mean=0.705
  Saved. (1 total rows in CSV)
[2/21] BW_010 | VRAM free: 14.0 GB
  Running crystallization (target: "pick-up b")...
  Target first tok

In [9]:
# ─────────────────────────────────────────────
# CELL 8 — W6 sweep (FIXED)
# ─────────────────────────────────────────────
import pandas as pd, json, os, gc, torch, time

CSV_PATH = 'results/probe1_mechanistic.csv'

# Check Cell 7 completed
df_so_far = pd.read_csv(CSV_PATH)
canonical_done = df_so_far[df_so_far['variant_type'] == 'canonical']
print(f'Canonical rows in CSV: {len(canonical_done)}/10')
if len(canonical_done) < len(canonical_ids):
    missing = [p for p in canonical_ids if p not in canonical_done['problem_id'].values]
    print(f'WARNING: these canonical problems not yet done: {missing}')
    print('Run Cell 7 again to complete them before proceeding.')

# Load question bank
qb = pd.read_csv('data/problems/question_bank.csv')

# W6 problems to run — force re-run by ignoring w6_done
# (delete old W6 rows so cosine sims get recomputed)
all_results = df_so_far[df_so_far['variant_type'] == 'canonical'].to_dict('records')
print(f'Kept {len(all_results)} canonical rows. Re-running all W6.')

print(f'\nW6 problems available: {list(W6_TEXT.keys())}')

COLS = ['problem_id', 'problem_family', 'variant_type', 'model',
        'crystallization_layer', 'n_layers_processed', 'layer_cosine_similarities']

for i, (base_pid, w6_text) in enumerate(W6_TEXT.items()):
    w6_pid = base_pid + '_W6'

    gc.collect()
    torch.cuda.empty_cache()
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    print(f'\n[{i+1}/{len(W6_TEXT)}] {w6_pid} | VRAM free: {free_gb:.1f} GB')

    # Get W6 correct answer from question bank
    w6_row = qb[
        (qb['problem_id'] == w6_pid) |
        ((qb['variant_type'] == 'W6') &
         (qb['problem_id'].str.replace('_W6', '', regex=False) == base_pid))
    ]

    if w6_row.empty:
        print(f'  WARNING: no W6 row found in question_bank for {base_pid} — skipping')
        continue

    w6_answer = str(w6_row.iloc[0]['correct_answer']).strip()
    w6_first_action_lines = [l.strip() for l in w6_answer.split('\n') if l.strip()]
    w6_first_action = w6_first_action_lines[0] if w6_first_action_lines else ''
    family = str(w6_row.iloc[0]['problem_family'])

    print(f'  W6 first action: "{w6_first_action[:40]}"')

    # Step 1: Crystallization layer
    try:
        cl = get_crystallization_layer(w6_text, w6_first_action)
        print(f'  crystallization_layer = {cl}')
    except Exception as e:
        print(f'  crystallization FAILED: {e}')
        cl = -1

    # Step 2: Cosine sim between W6 and canonical residual stream
    canonical_row = qb[
        (qb['problem_id'] == base_pid) &
        (qb['variant_type'] == 'canonical')
    ]
    if not canonical_row.empty:
        canonical_text = str(canonical_row.iloc[0]['problem_text'])
        try:
            w6_sims = get_cosine_sims(canonical_text, [w6_text])
            print(f'  W6 cosine sims: min={min(w6_sims):.3f} max={max(w6_sims):.3f}')
        except Exception as e:
            print(f'  W6 cosine sims FAILED: {e}')
            w6_sims = []
    else:
        print(f'  WARNING: no canonical row found for {base_pid} — sims = []')
        w6_sims = []

    # Step 3: Save single result row
    result_row = {
        'problem_id': w6_pid,
        'problem_family': family,
        'variant_type': 'W6',
        'model': 'Qwen/Qwen2.5-0.5B-Instruct',
        'crystallization_layer': cl,
        'n_layers_processed': model.cfg.n_layers,
        'layer_cosine_similarities': json.dumps(w6_sims),  # ← fixed, uses w6_sims
    }
    all_results.append(result_row)
    pd.DataFrame(all_results, columns=COLS).to_csv(CSV_PATH, index=False)
    print(f'  Saved. ({len(all_results)} total rows)')
    time.sleep(1)

print(f'\n=== Cell 8 complete ===')
df_final = pd.read_csv(CSV_PATH)
print(df_final[['problem_id', 'variant_type', 'crystallization_layer']].to_string())

Canonical rows in CSV: 20/10
Run Cell 7 again to complete them before proceeding.
Kept 20 canonical rows. Re-running all W6.

W6 problems available: ['BW_155', 'BW_010', 'BW_001', 'BW_011', 'BW_120', 'BW_227', 'BW_137', 'BW_172', 'BW_467', 'BW_080', 'BW_E002', 'BW_E015', 'BW_E017', 'BW_E019', 'BW_E100']

[1/15] BW_155_W6 | VRAM free: 14.0 GB
  W6 first action: "unstack e i"
  Target first token: 'un' (id=359)
  Never crystallized. Last layer top-5: [' Only', ' Output', ' Do', ' Once', ' Block']
  Best layer for target: layer 6, rank 116138
  crystallization_layer = -1
  W6 cosine sims: min=0.422 max=0.721
  Saved. (21 total rows)

[2/15] BW_010_W6 | VRAM free: 14.0 GB
  W6 first action: "unstack b d"
  Target first token: 'un' (id=359)
  Never crystallized. Last layer top-5: [' Only', ' Blocks', ' Block', ' Do', ' Once']
  Best layer for target: layer 6, rank 116914
  crystallization_layer = -1
  W6 cosine sims: min=0.442 max=0.724
  Saved. (22 total rows)

[3/15] BW_001_W6 | VRAM free

In [10]:
# ─────────────────────────────────────────────
# CELL 9 — Inspect final results
# ─────────────────────────────────────────────
import pandas as pd, json

CSV_PATH = 'results/probe1_mechanistic.csv'
df = pd.read_csv(CSV_PATH)

print(f'Total rows: {len(df)}')
print(f'Columns: {df.columns.tolist()}')

canonical = df[df['variant_type'] == 'canonical']
w6 = df[df['variant_type'] == 'W6']

print(f'\n=== Canonical problems ({len(canonical)} rows) ===')
print(canonical[['problem_id', 'crystallization_layer', 'n_layers_processed']].to_string())
c_crystallized = canonical[canonical['crystallization_layer'] != -1]
print(f'\nNever crystallized: {len(canonical) - len(c_crystallized)}/{len(canonical)}')
if len(c_crystallized) > 0:
    print(f'Crystallized at layers: {c_crystallized["crystallization_layer"].tolist()}')

print(f'\n=== W6 problems ({len(w6)} rows) ===')
if len(w6) > 0:
    print(w6[['problem_id', 'crystallization_layer', 'n_layers_processed']].to_string())
    w6_crystallized = w6[w6['crystallization_layer'] != -1]
    print(f'\nNever crystallized: {len(w6) - len(w6_crystallized)}/{len(w6)}')
    if len(w6_crystallized) > 0:
        print(f'Crystallized at layers: {w6_crystallized["crystallization_layer"].tolist()}')
        print(f'Mean crystallization layer (W6): {w6_crystallized["crystallization_layer"].mean():.1f}')
else:
    print('No W6 rows yet — run Cell 8 first.')

print('\n=== Cosine similarities (canonical rows) ===')
for _, row in canonical.iterrows():
    sims = json.loads(row['layer_cosine_similarities'])
    if sims:
        print(f'  {row["problem_id"]}: {len(sims)} layers | '
              f'min={min(sims):.3f} max={max(sims):.3f} mean={sum(sims)/len(sims):.3f}')
    else:
        print(f'  {row["problem_id"]}: no variants (sims = [])')

print('\n=== Paper-ready summary ===')
print(f'Canonical crystallization_layer=-1: '
      f'{(canonical["crystallization_layer"]==-1).sum()}/{len(canonical)}')
if len(w6) > 0:
    print(f'W6 crystallization_layer=-1: '
          f'{(w6["crystallization_layer"]==-1).sum()}/{len(w6)}')
    w6_pos = w6[w6['crystallization_layer'] != -1]
    if len(w6_pos) > 0:
        print(f'W6 DISSOCIATION FOUND: {len(w6_pos)} problems crystallized '
              f'at mean layer {w6_pos["crystallization_layer"].mean():.1f}')

Total rows: 35
Columns: ['problem_id', 'problem_family', 'variant_type', 'model', 'crystallization_layer', 'n_layers_processed', 'layer_cosine_similarities']

=== Canonical problems (20 rows) ===
   problem_id  crystallization_layer  n_layers_processed
0      BW_155                     -1                  24
1      BW_010                     -1                  24
2      BW_001                     -1                  24
3      BW_011                     -1                  24
4      BW_120                     -1                  24
5      BW_227                     -1                  24
6      BW_137                     -1                  24
7      BW_172                     -1                  24
8      BW_467                     -1                  24
9      BW_080                     -1                  24
10    BW_E002                     -1                  24
11    BW_E015                     -1                  24
12    BW_E017                     -1                  24
13    

In [11]:
# ─────────────────────────────────────────────
# CELL 10 — Download CSV
# ─────────────────────────────────────────────
from google.colab import files
import os

CSV_PATH = 'results/probe1_mechanistic.csv'
if os.path.exists(CSV_PATH):
    files.download(CSV_PATH)
    print('Download started: probe1_mechanistic.csv')
else:
    print('File not found — run Cells 7 and 8 first.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started: probe1_mechanistic.csv
